## <center>Khalifa University — Agentic AI Program</center>
### <center>Program 5 · Day 1: Advanced Agentic Workflows</center>
# Exercise 2: CrewAI — Agents, Tasks, and Context

**Module:** Declarative Multi-Agent Frameworks  
**Companion Script:** `Day 1/Excercises/day1_workbook_code/patient_crewai.py`  
**Estimated Practical Time:** 40 minutes  
**Target Model:** Local Ollama (`qwen3.5:4b`, `qwen3:4b-instruct`, `qwen2.5:7b`, or `qwen2.5:3b`)

---

### Aim & Learning Objectives
By the end of this exercise, you will be able to:
1. **Transition from procedural Python functions** to a declarative multi-agent framework (**CrewAI**).
2. **Define specialized Agents** with concrete roles, goals, and backstories while disabling delegation (`allow_delegation=False`).
3. **Wire Task dependencies** using the `context` parameter to form a sequential Directed Acyclic Graph (DAG).
4. **Supply ground-truth authority** directly to the reviewer task to prevent hallucination cascading.
5. **Execute a sequential crew** (`Process.sequential`) with standard `crew.kickoff()` / `crew.kickoff_async()` and inspect individual task outputs.

## CrewAI Architecture & Context Flow

In CrewAI, workflows are orchestrated by assigning `Task` objects to `Agent` objects, then executing them through a `Crew` manager. Rather than passing function outputs manually, we wire dependencies using `context=[...]` to form a clean DAG (Directed Acyclic Graph):

<p align="center">
  <img src="images/exercise2_crewai_flow.jpg" alt="Exercise 2: CrewAI Architecture & Context Flow" style="max-width: 100%; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.08);">
</p>

### Agent Configuration Matrix

| Agent Name | Role | Goal | Backstory Persona | `allow_delegation` | Input Source | Primary Responsibility |
|---|---|---|---|---|---|---|
| **Extractor** | Patient fact extractor | Extract only facts explicitly recorded in the note | You organize notes and never infer missing facts. | `False` | `PATIENT_NOTE` | Extract 8 clinical fields into bullets without extrapolation. |
| **Writer** | Patient report writer | Write an accurate report from the supplied facts | You add no diagnosis, stage, or treatment advice. | `False` | `context=[extract_task]` | Synthesize concise clinical narrative under 150 words. |
| **Reviewer** | Patient report reviewer | Find differences between the report and source note | You check omissions, errors, and unsupported claims. | `False` | `context=[write_task]` + `PATIENT_NOTE` | Audit draft against original ground-truth source note. |

### Sequential Execution Flow & Task Lifecycle Matrix

| Execution Phase | Task Object | Assigned Agent | Input Context / Data | Produced Artifact | Downstream Dependency | Termination Rule |
|---|---|---|---|---|---|---|
| **Phase 1: Extraction** | `extract_task` | `Extractor Agent` | `PATIENT_NOTE` (via task description) | 8-point factual bullet list | Output forwarded via `context=[extract_task]` | Completes extraction task |
| **Phase 2: Synthesis** | `write_task` | `Writer Agent` | `context=[extract_task]` | Clinical report draft (<150 words) | Output forwarded via `context=[write_task]` | Completes writing task |
| **Phase 3: Verification** | `review_task` | `Reviewer Agent` | `context=[write_task]` + `PATIENT_NOTE` (direct authority) | `PASS` or `REVISE` with line-item corrections | Stored in `review_task.output` | Completes review task |
| **Phase 4: Inspection** | Output Harvest | Python Orchestrator | Iterates `[extract_task, write_task, review_task]` | Multi-task execution outputs | Console display & human review | Crew execution finished |

### Why `allow_delegation=False` is Essential
In multi-agent pipelines where strict sequential ordering is required, enabling delegation allows agents to spawn unplanned sub-queries, ask peers for assistance, or skip assigned tasks. Setting `allow_delegation=False` forces CrewAI to respect the deterministic pipeline order.

## Prerequisites & Configuration
This exercise requires the `crewai` library:
```bash
pip install "crewai[litellm]"
```
Ollama must be running locally at `http://localhost:11434`.

In [1]:
import os
import sys

# Opt out of CrewAI telemetry prompt
os.environ["CREWAI_TELEMETRY_OPT_OUT"] = "true"

MODEL_NAME = os.environ.get("OLLAMA_MODEL", "qwen3:4b-instruct")
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")

print(f"Target Model: {MODEL_NAME}")
print(f"Ollama URL:   {OLLAMA_URL}")

Target Model: qwen3:4b-instruct
Ollama URL:   http://localhost:11434


## Preflight Check
Verifies that `crewai` is installed, tests the Ollama connection, and auto-selects the active model.

In [2]:
import importlib.util
import json
import urllib.request

if importlib.util.find_spec("crewai") is None:
    raise RuntimeError("Missing 'crewai'. Install with: pip install crewai")

try:
    with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=3) as resp:
        data = json.loads(resp.read().decode())
        installed = [m.get("name") for m in data.get("models", [])]
    print(f"CrewAI ready. Ollama server active with {len(installed)} models.")
    
    # Auto-match model
    if not os.environ.get("OLLAMA_MODEL"):
        if any(m.startswith(MODEL_NAME) or MODEL_NAME.startswith(m.split(":")[0]) for m in installed):
            pass
        elif any("qwen3.5:4b" in m for m in installed):
            MODEL_NAME = next(m for m in installed if "qwen3.5:4b" in m)
        elif any("qwen2.5:7b" in m for m in installed):
            MODEL_NAME = next(m for m in installed if "qwen2.5:7b" in m)
        elif any("qwen2.5:3b" in m for m in installed):
            MODEL_NAME = next(m for m in installed if "qwen2.5:3b" in m)
        elif installed:
            MODEL_NAME = installed[0]
    print(f"Active model for CrewAI: '{MODEL_NAME}'")
except Exception as e:
    print(f"Warning: Could not connect to Ollama: {e}")

CrewAI ready. Ollama server active with 3 models.
Active model for CrewAI: 'qwen3:4b-instruct'


## The Fictional Patient Note (Ground Truth)

The patient note below serves as the single clinical source of truth for the entire workflow.

In [3]:
PATIENT_NOTE = """
Patient ID: DEMO-001
Age: 60 years
Diagnosis recorded by the clinician: Colon cancer
Reported symptoms: Abdominal pain and blood in the stool
Colonoscopy finding: A mass in the colon
Biopsy result: Adenocarcinoma
Cancer stage: Not provided
Treatment plan: Not provided
""".strip()

print("Loaded Fictional Patient Note (Source of Truth):")
print("-" * 50)
print(PATIENT_NOTE)
print("-" * 50)

Loaded Fictional Patient Note (Source of Truth):
--------------------------------------------------
Patient ID: DEMO-001
Age: 60 years
Diagnosis recorded by the clinician: Colon cancer
Reported symptoms: Abdominal pain and blood in the stool
Colonoscopy finding: A mass in the colon
Biopsy result: Adenocarcinoma
Cancer stage: Not provided
Treatment plan: Not provided
--------------------------------------------------


## Step 1: Configure LLM & Define the Three Agents

In CrewAI, each participant in the workflow is an `Agent` defined by:
- **`role`**: What the agent is (e.g., `"Patient fact extractor"`).
- **`goal`**: The objective the agent must achieve.
- **`backstory`**: Context and persona guidelines that steer prompt generation.
- **`allow_delegation=False`**: Prevents workers from dynamically spawning new tasks or delegating work to peers, preserving a deterministic sequence.

In [4]:
from crewai import Agent, Crew, LLM, Process, Task

# Configure local LLM client for CrewAI
crew_llm = LLM(
    model=f"ollama/{MODEL_NAME}",
    base_url=OLLAMA_URL,
    temperature=0,
)

# 1. Extractor Agent
extractor = Agent(
    role="Patient fact extractor",
    goal="Extract only facts explicitly recorded in the note.",
    backstory="You organize notes and never infer missing facts.",
    llm=crew_llm,
    allow_delegation=False,
)

# 2. Writer Agent
writer = Agent(
    role="Patient report writer",
    goal="Write an accurate report from the supplied facts.",
    backstory="You add no diagnosis, stage, or treatment advice.",
    llm=crew_llm,
    allow_delegation=False,
)

# 3. Reviewer Agent
reviewer = Agent(
    role="Patient report reviewer",
    goal="Find differences between the report and source note.",
    backstory="You check omissions, errors, and unsupported claims.",
    llm=crew_llm,
    allow_delegation=False,
)

print("CrewAI Agents configured:")
print(f"  - {extractor.role}")
print(f"  - {writer.role}")
print(f"  - {reviewer.role}")

CrewAI Agents configured:
  - Patient fact extractor
  - Patient report writer
  - Patient report reviewer


## Step 2: Define Tasks and Context Handoffs

Notice how information flows between tasks:
- `extract_task`: Receives `PATIENT_NOTE` directly in its description.
- `write_task`: Uses `context=[extract_task]` to automatically receive the extractor's output.
- `review_task`: Uses `context=[write_task]` to receive the written draft, **AND** receives the original source note in its description as the sole factual authority.

In [5]:
extract_task = Task(
    description=(
        "Extract patient ID, age, recorded diagnosis, symptoms, "
        "colonoscopy, biopsy, cancer stage, and treatment plan. "
        "Use only the note. Write Not provided for missing facts.\n"
        f"SOURCE NOTE:\n{PATIENT_NOTE}"
    ),
    expected_output="A bullet list with all eight fields.",
    agent=extractor,
)

write_task = Task(
    description=(
        "Write a summary under 150 words from the extracted facts. "
        "Include all eight fields. State when stage and treatment "
        "are not provided. Add no conclusions or recommendations. "
        "Say 'recorded diagnosis'; do not infer how or when the "
        "diagnosis was made or connect it causally to symptoms/tests."
    ),
    expected_output="A short patient report using only the facts.",
    agent=writer,
    context=[extract_task],
)

review_task = Task(
    description=(
        "Compare the draft in your context with the original note. "
        "Check all eight fields for omissions and incorrect facts, "
        "including invented stage or treatment. Reject invented "
        "explanations of how or when diagnosis was made. Return PASS if "
        "there are no issues. Otherwise return REVISE and specific "
        "corrections. Do not rewrite the report. "
        "ORIGINAL NOTE is the sole authority; the draft can be wrong. "
        "Never fill gaps in the source using draft claims. For each "
        "issue, quote the wrong claim and state its source-supported "
        "correction. Missing stage/treatment must stay 'Not provided'.\n"
        f"ORIGINAL NOTE:\n{PATIENT_NOTE}"
    ),
    expected_output="PASS or REVISE with specific corrections.",
    agent=reviewer,
    context=[write_task],
)

print("CrewAI Tasks defined and chained via context parameter.")

CrewAI Tasks defined and chained via context parameter.


## Step 3: Run the Sequential Crew & Inspect Outputs

We assemble the agents and tasks into a `Crew` with `process=Process.sequential` and execute via `patient_crew.kickoff()` / `kickoff_async()`.

In [6]:
patient_crew = Crew(
    agents=[extractor, writer, reviewer],
    tasks=[extract_task, write_task, review_task],
    process=Process.sequential,
    verbose=True,
)

print("Kicking off CrewAI execution...")
# In Jupyter notebooks with an active event loop, kickoff_async() is required.
# In standalone Python scripts, kickoff() is used.
import asyncio
try:
    if asyncio.get_running_loop():
        crew_output = await patient_crew.kickoff_async()
except RuntimeError:
    crew_output = patient_crew.kickoff()

print("\n" + "=" * 50)
print("CREW EXECUTION COMPLETE - INDIVIDUAL TASK OUTPUTS:")
print("=" * 50)

for label, task in [
    ("FACTS", extract_task),
    ("REPORT", write_task),
    ("REVIEW", review_task),
]:
    print(f"\n--- {label} ---")
    print(task.output.raw if task.output else "No output produced")

print("\nClassroom draft only; human review is needed.")

Kicking off CrewAI execution...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0f24bea5-40a3-4f7a-923f-6551e0c09609                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Extract patient ID, age, recorded diagnosis, symptoms, colonoscopy, biopsy, cancer stage, and treatment  │
│  plan. Use only the note. Write Not provided for missing facts.                                                 │
│  SOURCE NOTE:                                                                                                   │
│  Patient ID: DEMO-001                                                                                           │
│  Age: 60 years                                                                                                  │
│  Diagnosis recorded by the clinician: Colon cancer                                                              │
│  Reported symptoms: Abdominal pain and blood in the stool                                                       │
│  Colonoscopy finding: A mass in the colon                                                                       │
│  Biopsy result: Adenocarcinoma                                                                                  │
│  Cancer stage: Not provided                                                                                     │
│  Treatment plan: Not provided                                                                                   │
│  ID: f9eac8f3-c34a-40e6-9afa-4ce0e4ccc6eb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Patient fact extractor                                                                                  │
│                                                                                                                 │
│  Task: Extract patient ID, age, recorded diagnosis, symptoms, colonoscopy, biopsy, cancer stage, and treatment  │
│  plan. Use only the note. Write Not provided for missing facts.                                                 │
│  SOURCE NOTE:                                                                                                   │
│  Patient ID: DEMO-001                                                                                           │
│  Age: 60 years                                                                                                  │
│  Diagnosis recorded by the clinician: Colon cancer                                                              │
│  Reported symptoms: Abdominal pain and blood in the stool                                                       │
│  Colonoscopy finding: A mass in the colon                                                                       │
│  Biopsy result: Adenocarcinoma                                                                                  │
│  Cancer stage: Not provided                                                                                     │
│  Treatment plan: Not provided                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Patient fact extractor                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  - Patient ID: DEMO-001                                                                                         │
│  - Age: 60 years                                                                                                │
│  - Recorded diagnosis: Colon cancer                                                                             │
│  - Symptoms: Abdominal pain and blood in the stool                                                              │
│  - Colonoscopy: A mass in the colon                                                                             │
│  - Biopsy: Adenocarcinoma                                                                                       │
│  - Cancer stage: Not provided                                                                                   │
│  - Treatment plan: Not provided                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Extract patient ID, age, recorded diagnosis, symptoms, colonoscopy, biopsy, cancer stage, and treatment  │
│  plan. Use only the note. Write Not provided for missing facts.                                                 │
│  SOURCE NOTE:                                                                                                   │
│  Patient ID: DEMO-001                                                                                           │
│  Age: 60 years                                                                                                  │
│  Diagnosis recorded by the clinician: Colon cancer                                                              │
│  Reported symptoms: Abdominal pain and blood in the stool                                                       │
│  Colonoscopy finding: A mass in the colon                                                                       │
│  Biopsy result: Adenocarcinoma                                                                                  │
│  Cancer stage: Not provided                                                                                     │
│  Treatment plan: Not provided                                                                                   │
│  Agent: Patient fact extractor                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (6 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Patient report reviewer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  REVISE                                                                                                         │
│                                                                                                                 │
│  Wrong claim: "Diagnosis recorded by the clinician: Colon cancer"                                               │
│  Source-supported correction: "Recorded diagnosis: Colon cancer" — the original note does not state that the    │
│  diagnosis was "recorded by the clinician"; this is an invented attribution. The original note only states      │
│  "Recorded diagnosis: Colon cancer", so the claim about the clinician recording it is unsupported and must be   │
│  removed.                                                                                                       │
│                                                                                                                 │
│  Wrong claim: "Reported symptoms: Abdominal pain and blood in the stool"                                        │
│  Source-supported correction: This claim is correct and matches the original note.                              │
│                                                                                                                 │
│  Wrong claim: "Colonoscopy finding: A mass in the colon"                                                        │
│  Source-supported correction: This claim is correct and matches the original note.                              │
│                                                                                                                 │
│  Wrong claim: "Biopsy result: Adenocarcinoma"                                                                   │
│  Source-supported correction: This claim is correct and matches the original note.                              │
│                                                                                                                 │
│  Wrong claim: "Cancer stage: Not provided"                                                                      │
│  Source-supported correction: This claim is correct and matches the original note.                              │
│                                                                                                                 │
│  Wrong claim: "Treatment plan: Not provided"                                                                    │
│  Source-supported correction: This claim is correct and matches the original note.                              │
│                                                                                                                 │
│  Final correction: The draft incorrectly attributes the diagnosis to being "recorded by the clinician", which   │
│  is not stated in the original note. The original note only states "Recorded diagnosis: Colon cancer", without  │
│  specifying who recorded it. This invented detail must be removed.                                              │
│                                                                                                                 │
│  Corrected version (as required, without rewriting the report):                                                 │
│                                                                                                                 │
│  Patient ID: DEMO-001                                  

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Compare the draft in your context with the original note. Check all eight fields for omissions and       │
│  incorrect facts, including invented stage or treatment. Reject invented explanations of how or when diagnosis  │
│  was made. Return PASS if there are no issues. Otherwise return REVISE and specific corrections. Do not         │
│  rewrite the report. ORIGINAL NOTE is the sole authority; the draft can be wrong. Never fill gaps in the        │
│  source using draft claims. For each issue, quote the wrong claim and state its source-supported correction.    │
│  Missing stage/treatment must stay 'Not provided'.                                                              │
│  ORIGINAL NOTE:                                                                                                 │
│  Patient ID: DEMO-001                                                                                           │
│  Age: 60 years                                                                                                  │
│  Diagnosis recorded by the clinician: Colon cancer                                                              │
│  Reported symptoms: Abdominal pain and blood in the stool                                                       │
│  Colonoscopy finding: A mass in the colon                                                                       │
│  Biopsy result: Adenocarcinoma                                                                                  │
│  Cancer stage: Not provided                                                                                     │
│  Treatment plan: Not provided                                                                                   │
│  Agent: Patient report reviewer                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


CREW EXECUTION COMPLETE - INDIVIDUAL TASK OUTPUTS:

--- FACTS ---
- Patient ID: DEMO-001
- Age: 60 years
- Recorded diagnosis: Colon cancer
- Symptoms: Abdominal pain and blood in the stool
- Colonoscopy: A mass in the colon
- Biopsy: Adenocarcinoma
- Cancer stage: Not provided
- Treatment plan: Not provided

--- REPORT ---
Patient ID: DEMO-001  
Age: 60 years  
Recorded diagnosis: Colon cancer  
Symptoms: Abdominal pain and blood in the stool  
Colonoscopy: A mass in the colon  
Biopsy: Adenocarcinoma  
Cancer stage: Not provided  
Treatment plan: Not provided

--- REVIEW ---
REVISE

Wrong claim: "Diagnosis recorded by the clinician: Colon cancer"  
Source-supported correction: "Recorded diagnosis: Colon cancer" — the original note does not state that the diagnosis was "recorded by the clinician"; this is an invented attribution. The original note only states "Recorded diagnosis: Colon cancer", so the claim about the clinician recording it is unsupported and must be removed.

Wrong c

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0f24bea5-40a3-4f7a-923f-6551e0c09609                                                                       │
│  Final Output: REVISE                                                                                           │
│                                                                                                                 │
│  Wrong claim: "Diagnosis recorded by the clinician: Colon cancer"                                               │
│  Source-supported correction: "Recorded diagnosis: Colon cancer" — the original note does not state that the    │
│  diagnosis was "recorded by the clinician"; this is an invented attribution. The original note only states      │
│  "Recorded diagnosis: Colon cancer", so the claim about the clinician recording it is unsupported and must be   │
│  removed.                                                                                                       │
│                                                                                                                 │
│  Wrong claim: "Reported symptoms: Abdominal pain and blood in the stool"                                        │
│  Source-supported correction: This claim is correct and matches the original note.                              │
│                                                                                                                 │
│  Wrong claim: "Colonoscopy finding: A mass in the colon"                                                        │
│  Source-supported correction: This claim is correct and matches the original note.                              │
│                                                                                                                 │
│  Wrong claim: "Biopsy result: Adenocarcinoma"                                                                   │
│  Source-supported correction: This claim is correct and matches the original note.                              │
│                                                                                                                 │
│  Wrong claim: "Cancer stage: Not provided"                                                                      │
│  Source-supported correction: This claim is correct and matches the original note.                              │
│                                                                                                                 │
│  Wrong claim: "Treatment plan: Not provided"                                                                    │
│  Source-supported correction: This claim is correct and matches the original note.                              │
│                                                                                                                 │
│  Final correction: The draft incorrectly attributes the diagnosis to being "recorded by the clinician", which   │
│  is not stated in the original note. The original note only states "Recorded diagnosis: Colon cancer", without  │
│  specifying who recorded it. This invented detail must be removed.                                              │
│                                                                                                                 │
│  Corrected version (as required, without rewriting the report):                                                 │
│                                                                                                                 │
│  Patient ID: DEMO-001                                 

## Exercise 2 Checkpoint & Key Insights

Review your run and verify your understanding of the CrewAI abstractions:

1. **Which task receives the extractor output through `context`?**  
   *Answer:* `write_task` receives `context=[extract_task]`.

2. **What could happen if the reviewer task did not receive the original note?**  
   *Answer:* If the reviewer only saw the draft and extracted facts, any omission or invention introduced during extraction would be invisible to the reviewer.

3. **How does `Process.sequential` differ from plain Python?**  
   *Answer:* CrewAI internally orchestrates prompt construction, agent memory injection, and context forwarding across tasks, whereas plain Python uses explicit function calls and variable arguments.

4. **Summary of Key Configuration Primitives:**
   - `llm=llm`: All three agents share one unified model configuration.
   - `agent=extractor`: Assigns a specific task to its worker role.
   - `context=[extract_task]`: Passes extracted facts forward to the writer.
   - `context=[write_task]`: Passes the draft report to the reviewer.
   - `Process.sequential`: Runs tasks strictly one after another in order.